# Exercise 1 - Blockchain Structure and Tamper Detection


In [ ]:
import hashlib
import json
import time

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def calculate_merkle_root(transactions):
    if not transactions:
        return sha256_text("")

    level = [sha256_text(tx) for tx in transactions]

    while len(level) > 1:
        # Duplicate the final hash when a level contains an odd number of nodes.
        if len(level) % 2 == 1:
            level.append(level[-1])

        level = [
            sha256_text(level[i] + level[i + 1])
            for i in range(0, len(level), 2)
        ]

    return level[0]

class Block:
    def __init__(self, index, transactions, previous_hash, timestamp=None):
        self.index = index
        self.timestamp = int(time.time()) if timestamp is None else timestamp
        self.transactions = list(transactions)
        self.previous_hash = previous_hash
        self.merkle_root = calculate_merkle_root(self.transactions)
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        # Transactions are represented by the Merkle Root in the block header.
        header = {
            "index": self.index,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "merkle_root": self.merkle_root,
        }
        return hashlib.sha256(
            json.dumps(header, sort_keys=True).encode("utf-8")
        ).hexdigest()

    def as_dict(self):
        return {
            "block": self.index,
            "transactions": self.transactions,
            "merkle_root": self.merkle_root,
            "previous_hash": self.previous_hash,
            "hash": self.hash,
        }

class Blockchain:
    def __init__(self):
        self.chain = [
            Block(
                index=1,
                transactions=["Genesis Block"],
                previous_hash="0" * 64,
            )
        ]

    def add_block(self, transactions):
        previous_block = self.chain[-1]
        new_block = Block(
            index=len(self.chain) + 1,
            transactions=transactions,
            previous_hash=previous_block.hash,
        )
        self.chain.append(new_block)

    def validate(self):
        for position, block in enumerate(self.chain):
            recomputed_merkle = calculate_merkle_root(block.transactions)

            if block.merkle_root != recomputed_merkle:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Merkle Root mismatch",
                    "stored": block.merkle_root,
                    "recomputed": recomputed_merkle,
                    "blocks_checked": position + 1,
                }

            recomputed_hash = block.calculate_hash()
            if block.hash != recomputed_hash:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Block hash mismatch",
                    "stored": block.hash,
                    "recomputed": recomputed_hash,
                    "blocks_checked": position + 1,
                }

            expected_previous = "0" * 64 if position == 0 else self.chain[position - 1].hash
            if block.previous_hash != expected_previous:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Previous-hash link mismatch",
                    "stored": block.previous_hash,
                    "recomputed": expected_previous,
                    "blocks_checked": position + 1,
                }

        return {
            "valid": True,
            "first_failing_block": None,
            "reason": "No integrity errors detected",
            "stored": None,
            "recomputed": None,
            "blocks_checked": len(self.chain),
        }

    def print_chain(self):
        for block in self.chain:
            print(json.dumps(block.as_dict(), indent=2))
            print("-" * 88)


In [ ]:
blockchain = Blockchain()

transaction_sets = [
    ["Alice -> Bob: 5 BTC", "Bob -> Charlie: 1 BTC"],
    ["Charlie -> David: 2 BTC", "Eve -> Alice: 3 BTC"],
    ["David -> Alice: 0.5 BTC", "Bob -> Eve: 0.25 BTC"],
    ["Alice -> Eve: 1.2 BTC", "Charlie -> Bob: 0.8 BTC"],
    ["Eve -> David: 0.7 BTC", "David -> Bob: 0.4 BTC"],
    ["Bob -> Alice: 1.1 BTC", "Alice -> Charlie: 0.3 BTC"],
    ["Charlie -> Eve: 0.9 BTC", "Eve -> Bob: 0.2 BTC"],
    ["David -> Charlie: 1.5 BTC", "Bob -> David: 0.6 BTC"],
    ["Alice -> David: 0.75 BTC", "Eve -> Charlie: 0.45 BTC"],
]

for transactions in transaction_sets:
    blockchain.add_block(transactions)

assert len(blockchain.chain) == 10

print("=== BLOCKCHAIN BEFORE MODIFICATION ===")
blockchain.print_chain()

print("=== VALIDATION BEFORE MODIFICATION ===")
print(json.dumps(blockchain.validate(), indent=2))


In [ ]:
# Required tampering test:
# Change transaction data in the 5th block, but DO NOT recalculate its stored Merkle Root or hash.
fifth_block = blockchain.chain[4]
original_transaction = fifth_block.transactions[0]
fifth_block.transactions[0] = "ATTACKER -> ATTACKER: 999 BTC"

print("Original transaction:", original_transaction)
print("Modified transaction:", fifth_block.transactions[0])

print("\n=== MODIFIED 5TH BLOCK ===")
print(json.dumps(fifth_block.as_dict(), indent=2))

print("\n=== VALIDATION AFTER MODIFICATION ===")
tamper_result = blockchain.validate()
print(json.dumps(tamper_result, indent=2))


## Reflection Questions

### 1. Why does modifying Block 5 affect later blocks?

The transaction change alters Block 5's recomputed Merkle Root. Because the stored Merkle Root remains unchanged, validation immediately detects a **Merkle Root mismatch at Block 5**. The block hash is calculated from the Merkle Root and the other header fields, so a correctly recalculated Block 5 would also have a different hash. Block 6 stores the original Block 5 hash in its `previous_hash` field, so Block 6 would then fail with a **previous-hash link mismatch**. This demonstrates that each block is cryptographically linked to the previous block.

In the required experiment, the attacker does not recalculate anything, so Block 5 is the first failure. If Block 5 were recalculated to hide the first mismatch, the stale link in Block 6 would expose the tampering.

### 2. What should a validation program report?

A useful validation report should include:

- **Whether the chain is valid:** gives an immediate integrity decision.
- **The first failing block:** identifies where corruption was first detected.
- **The failure reason:** distinguishes a Merkle Root mismatch, block hash mismatch, or previous-hash link mismatch.
- **Stored and recomputed values:** provides evidence for comparison and helps diagnose the change.
- **Number of blocks checked:** shows how far validation progressed before stopping.

The `validate()` method returns all of these fields. For the required tampering test, the result identifies Block 5 and reports a Merkle Root mismatch.

### 3. One improvement to resist tampering

A practical improvement would be to digitally sign each block header using the creator's private key and verify the signature with the corresponding public key. An attacker could still edit local transaction data, but they could not produce a valid replacement signature without the private key. Combining digital signatures with the existing SHA-256 hashes, Merkle Roots, and previous-hash links would provide both tamper detection and stronger proof of who created each block.
